In [1]:
import pinocchio as pin
import numpy as np
import time
import scipy
from example_robot_data import load

## visualise the robot
from pinocchio.visualize import MeshcatVisualizer

## visualise the polytope and the ellipsoid
import meshcat.geometry as g 

# import pycapacity 
import pycapacity as pycap

In [2]:
## from datasheet https://www.maxongroup.com/medias/sys_master/root/8882563907614/EN-21-300.pdf
# Coefficients for the motors
R_A1 = 35   # Coefficient for motor A related to pitch
R_A2 = 1.9  # Coefficient for motor A related to roll
R_B1 = 35   # Coefficient for motor B related to pitch
R_B2 = 1.9  # Coefficient for motor B related to roll

## ec45 flat 
## https://www.maxongroup.com/medias/sys_master/root/8882563907614/EN-21-300.pdf
current_nominal = 3.96 # Nominal current
rendement = 0.93 * 0.8
couple_nominal = 0.167 * rendement / current_nominal

# ecx42 flat
# https://file.notion.so/f/f/5e956a13-b7ed-4125-a93c-18d8732b4358/3d53b52a-b0cb-487e-9e57-8e14623172d0/page0261.pdf?table=block&id=10c6d6bd-0618-8046-bda7-faa6b48449fb&spaceId=5e956a13-b7ed-4125-a93c-18d8732b4358&expirationTimestamp=1728777600000&signature=ojA2Y657gRYIskTUe4cMN3Hu5be1HQAbqdiutzqfpgE&downloadName=ECX+FLAT+42+M.pdf
# current_nominal = 7.34 # Nominal current
# rendement = 0.93 * 0.8
# couple_nominal = 0.214 * rendement / current_nominal

J_o = np.array([[R_A2, R_B2], [R_A2, -R_B2]])
J_o_inv = np.linalg.inv(J_o)
Jo = scipy.linalg.block_diag(J_o,J_o)
Jo_inv = scipy.linalg.block_diag(J_o_inv,J_o_inv)

# orbita2d kinematics matrix
J_W = np.array([[R_A1 * R_A2, R_B1 * R_B2], [R_A1 * R_A2, -R_B1 * R_B2]])
J_W_inv = np.linalg.inv(J_W)
W = J_W*couple_nominal
W_inv = np.linalg.inv(W)
W_inv = scipy.linalg.block_diag(W_inv,W_inv)
W = scipy.linalg.block_diag(W,W)
JW_inv = scipy.linalg.block_diag(J_W_inv,J_W_inv)
JW = scipy.linalg.block_diag(J_W,J_W)

In [3]:

def computeCollisions(robot, geom_model, q, verbose = True):
    geom_data = geom_model.createData()
    has_collision = pin.computeCollisions(robot.model, robot.data, geom_model, geom_data, q, not verbose)
    if verbose:
        print("Has collision {}".format(has_collision))
        i = 0
        for k in range(len(geom_model.collisionPairs)): 
          cr = geom_data.collisionResults[k]
          cp = geom_model.collisionPairs[k]
          if cr.isCollision():
              i = i+1
              print("collision pair:",cp.first,",",cp.second,"- collision:","Yes" if cr.isCollision() else "No")
        print(i)
    return has_collision

# function visualising the polutope in meshcat
def visualise_polytope(q):
    # calculate the jacobian
    pin.framesForwardKinematics(robot.model,data,q)
    pin.computeJointJacobians(robot.model,data, q)
    J = pin.getFrameJacobian(robot.model, data, robot.model.getFrameId(robot.model.frames[-1].name), pin.LOCAL_WORLD_ALIGNED)
    # use only position jacobian
    J = J[:3,:]
    
    # end-effector pose
    Xee = data.oMf[robot.model.getFrameId(robot.model.frames[-1].name)]
    
    
    # calculate the polytope
    opt = {'calculate_faces':True}
    # calculate the polytope
    for_poly = pycap.robot.force_polytope(J, 2*t_min*R_A1*R_A2*couple_nominal, 2*t_max*R_A1*R_A2*couple_nominal,options=opt)
    # meshcat triangulated mesh
    poly = g.TriangularMeshGeometry(vertices=for_poly.vertices.T/500 + Xee.translation, faces=for_poly.face_indices)
    viz.viewer['poly'].set_object(poly, g.MeshBasicMaterial(color=0x0022ff, wireframe=True, linewidth=3, opacity=0.2))
    
    # calculate the polytope
    opt = {'calculate_faces':True}
    # calculate the polytope
    for_poly = pycap.robot.force_polytope(J@W_inv.T, t_min, t_max,options=opt)
    # meshcat triangulated mesh
    poly = g.TriangularMeshGeometry(vertices=for_poly.vertices.T/500 + Xee.translation, faces=for_poly.face_indices)
    viz.viewer['poly2'].set_object(poly, g.MeshBasicMaterial(color=0xff2200, wireframe=True, linewidth=3, opacity=0.2))


    for_poly_z = pycap.robot.force_polytope(J[2,:].reshape(1,-1), 2*t_min*R_A1*R_A2*couple_nominal, 2*t_max*R_A1*R_A2*couple_nominal,options=opt)
    for_poly1_z = pycap.robot.force_polytope((J@W_inv.T)[2,:].reshape(1,-1), t_min, t_max,options=opt)
    print(f"How many kilos can Reachy carry?\nindependent axis\t{np.max(for_poly_z.vertices)/ 9.81:.2f}kg\ncoupled axis\t\t{np.max(for_poly1_z.vertices)/ 9.81:.2f}kg" )

In [4]:
def compute_gravity(robot,q, keep = None):
    if keep is not None:
        return pin.computeGeneralizedGravity(robot.model, robot.data, q)[keep]
    else :
        return pin.computeGeneralizedGravity(robot.model, robot.data, q)

def carrying_capacity(robot, q, t_min, t_max ,tip = None, keep= None, verbose = None ):
    
    J, s = jacobian(robot,q,tip=tip,keep=keep)
    
    J = J[:3,:]
    
    g = pin.computeGeneralizedGravity(robot.model, robot.data, q)[keep]
    
    for_poly_z = pycap.robot.force_polytope((J@W_inv.T)[2,:].reshape(1,-1), t_min, t_max, t_bias = g@W_inv)
    # for_poly_z = pycap.robot.force_polytope((J@W_inv.T)[2,:].reshape(1,-1), 2*t_min*R_A1*R_A2*couple_nominal, 2*t_max*R_A1*R_A2*couple_nominal, t_bias = g)
    if verbose is not None:
        for_poly1_z = pycap.robot.force_polytope((J@W_inv.T)[2,:].reshape(1,-1), t_min, t_max)
        print(f"How many kilos can Reachy carry?\nwith gravity\t{np.max(for_poly_z.vertices)/ 9.81:.2f}kg\nno gravity\t\t{np.max(for_poly1_z.vertices)/ 9.81:.2f}kg" )
    
    return for_poly_z
    
def motor_torque_ratio_gravity(robot, q, tip= None, keep=None, verbose=None):
    
    # g = pin.computeGeneralizedGravity(robot.model, robot.data, q)[keep]
    # if mass is not None:
    J, s = jacobian(robot,q,tip=tip,keep=keep)
    if keep is not None:
        J = J[2,keep]
    else:
        J=J[2,:]
    g = W_inv@J.T
    
    g= np.abs(g)
    g_sh = np.sum(g[:2])/100
    g_el = np.sum(g[2:4])/100
    if verbose is not None:
        print("shoulder ratio:\n m1 {:0.2f}% m2 {:0.2f}%\nelbow ratio:\n m1 {:0.2f}% m2 {:0.2f}%".format(g[0]/g_sh,g[1]/g_sh, g[2]/g_el,g[3]/g_el)) 
    return g, [g[0]/g_sh,g[1]/g_sh, g[2]/g_el,g[3]/g_el]

In [5]:

def reduce_model(robot, tolock, lock_vals=None):
    # Get the ID of all existing joints
    lvals = np.zeros(robot.nq)
    jointsToLockIDs = []
    for i,jn in enumerate(tolock):
        if robot.model.existJointName(jn):
            jointsToLockIDs.append(robot.model.getJointId(jn))
            if lock_vals is not None:
                lvals[i] = lock_vals[i]
    model, collision_model = pin.buildReducedModel(robot.model, robot.collision_model, jointsToLockIDs, lvals)
    data = model.createData()
    return model, data, collision_model

def getJointNames(robot):
    names = []
    for j in robot.model.frames[1:]:
        if robot.index(j.name) <= robot.nq:
            names.append(j.name)
    return names


def config_collision_model(robot):
    model = robot.model
    data = robot.model.createData()
    geom_model = robot.collision_model
    for i in [0, 1, 2, 11, 20, 21]: # torso, head, 3x bars
        # geom_model.addCollisionPair(pin.CollisionPair(i,3)) # left upper arm
        # geom_model.addCollisionPair(pin.CollisionPair(i,4)) # left lower arm
        # geom_model.addCollisionPair(pin.CollisionPair(i,5)) # left wrist
        # geom_model.addCollisionPair(pin.CollisionPair(i,6)) # left palm
        geom_model.addCollisionPair(pin.CollisionPair(i,12)) # right upper arm
        geom_model.addCollisionPair(pin.CollisionPair(i,13)) #  right lower arm
        geom_model.addCollisionPair(pin.CollisionPair(i,14)) #right  wrist
        geom_model.addCollisionPair(pin.CollisionPair(i,15)) # left palm
    geom_data = pin.GeometryData(geom_model)
    robot.rebuildData()
    return geom_model, geom_data

def jacobian(robot, q, tip = None, keep= None):
        
    if tip is None:
        tip = robot.model.frames[-1].name
    joint_id = model.getFrameId(tip)
    # joint_id = robot.model.getFrameId("r_wrist_ball_link")
    # joint_id =  model.getFrameId(robot.model.frames[-1].name)
    J = pin.computeFrameJacobian(robot.model,
                                       robot.data,
                                       q,
                                       joint_id,
                                       reference_frame=pin.LOCAL_WORLD_ALIGNED)[:3,:]
    
    if keep is None:
        u,s,v = np.linalg.svd(J)
    else:
        u,s,v = np.linalg.svd(J[:,keep])
    # print(J)
    return J, s

def dk(robot, q, tip = None):
    if tip is None:
        tip = robot.model.frames[-1].name
    # joint_id =  robot.model.getFrameId(robot.model.frames[-1].name)
    joint_id = robot.model.getFrameId(tip)
    pin.framesForwardKinematics(robot.model, robot.data, q)
    return robot.data.oMf[robot.model.getFrameId(tip)].translation.copy(), robot.data.oMf[robot.model.getFrameId(tip)].rotation

In [6]:
# use reachy's right arm with only 4 joints for now as they are the contribute 
# to its force capacity the most
# urdf_path = "reachy_v3_fix.urdf"
urdf_path = "/home/gospar/pollen_robotics/reachy2_modelling/reachy_v3_fix_shoulder.urdf"
# urdf_path = 'reachy_v3_simple.urdf'
# urdf_path = 'reachy.urdf'
robot = pin.RobotWrapper.BuildFromURDF(urdf_path)
model, data = robot.model, robot.data
# lock the other joints
tolock = [
   "l_shoulder_pitch",
   "l_shoulder_roll",
   "l_elbow_yaw",
   "l_elbow_pitch",
   "l_wrist_roll",
   "l_wrist_pitch",
   "l_wrist_yaw",
   "l_hand_finger",
   "l_hand_finger_mimic",
   "l_hand_finger_proximal",
   "l_hand_finger_distal",
   "l_hand_finger_proximal_mimic",
   "l_hand_finger_distal_mimic",
   "neck_roll",
   "neck_pitch",
   "neck_yaw",
   # "r_shoulder_dummy_1",
   # "r_shoulder_dummy_2",
   # "r_shoulder_dummy_out",
   # "r_shoulder_pitch",
   # "r_shoulder_roll",
   # "r_elbow_yaw",
   # "r_elbow_pitch",
   "r_wrist_roll",
   "r_wrist_pitch",
   "r_wrist_yaw",
   "r_hand_finger",
   "r_hand_finger_mimic",
   "r_hand_finger_proximal",
   "r_hand_finger_distal",
   "r_hand_finger_proximal_mimic",
   "r_hand_finger_distal_mimic",
]

robot.model, robot.data, robot.collision_model = reduce_model(robot, tolock)
model, data = robot.model, robot.data
geom_model, geom_data = config_collision_model(robot)

In [7]:
from reachy2_symbolic_ik.symbolic_ik import SymbolicIK
import meshcat_shapes
import reachy2_modelling as r2

q_shoulder = {
    "beta": [10, 0, 15],
    "straight" : [0,0,0],
    "up10":[-10,0,0],
    "up20":[-20,0,0],
    "up20front10": [-20,0,10],
    "up20front15": [-20,0,15],
    "back5":[0,0,-5],
    "dvt":[-15, 0, 10]
}

tolock = [
   "r_shoulder_dummy_1",
   "r_shoulder_dummy_2",
   "r_shoulder_dummy_out"
   ]

# dvt
ik_sym_dvt = r2.symik.SymArm("r_arm", shoulder_offset=np.array(q_shoulder["dvt"]))
q_sh = q_shoulder["dvt"]
q_lock = [np.deg2rad(q_sh[0]-10), np.deg2rad(q_sh[2]-15), 0.0]
print(q_lock)
model, data, col_model = reduce_model(robot, tolock, q_lock)
robot = pin.RobotWrapper(model, col_model)
robot.rebuildData()

geom_model, geom_data = config_collision_model(robot)

q_min = np.array([-2*np.pi, -2.9, -2*np.pi, -2.22,-np.pi/4,-np.pi/4,-np.pi])[:robot.nq]
q_max = np.array([2*np.pi, 0.37, 2*np.pi, -0.1,np.pi/4,np.pi/4,np.pi])[:robot.nq]

robot.model.lowerPositionLimit = q_min
robot.model.upperPositionLimit = q_max

robot.model.velocityLimit = np.ones(robot.nq)*5
# get max velocity
t_max = np.ones(robot.nq)*4 # amps
t_min = -t_max

# Use robot configuration.
# q0 = np.random.uniform(q_min,q_max)
q = (q_min+q_max)/2


viz = MeshcatVisualizer(robot.model, robot.collision_model, robot.collision_model)
# Start a new MeshCat server and client.
viz.initViewer(open=False)
viz.loadViewerModel("reachy_dvt")#, color=[256,0,0,0.5])
viz.display(q)

viz.viewer.jupyter_cell()

/tmp/tmpjk40vw72


Unknown attribute "iyx" in /robot[@name='reachy2']/link[@name='base_link']/inertial/inertia
Unknown attribute "izx" in /robot[@name='reachy2']/link[@name='base_link']/inertial/inertia
Unknown attribute "izy" in /robot[@name='reachy2']/link[@name='base_link']/inertial/inertia
Scalar element defined multiple times: dynamics
Scalar element defined multiple times: dynamics
Unknown tag "ros2_control" in /robot[@name='reachy2']


[-0.4363323129985824, -0.08726646259971647, 0.0]
You can open the visualizer by visiting the following URL:
http://127.0.0.1:7000/static/


In [8]:
from ipywidgets import interact, FloatSlider
import meshcat_shapes
meshcat_shapes.frame(viz.viewer["end_effector_target"], opacity=0.5)

## from ipywidgets import interact, FloatSlider
kwargs = {'q[{}]'.format(i) : 
          FloatSlider(
              min = q_min[i], 
              max = q_max[i], 
              step = 0.01, 
              value = q[i]) 
          for i,q_1 in enumerate(q)}
@interact(**kwargs)
def update(**kwargs):
    global q
    q = np.array([v  for v in kwargs.values()])
    viz.display(q)
    dk(robot,q)
    print(compute_gravity(robot, q[:4]))
    viz.viewer["end_effector_target"].set_transform(robot.data.oMf[robot.model.getFrameId("r_arm_tip")].np)


interactive(children=(FloatSlider(value=0.0, description='q[0]', max=6.283185307179586, min=-6.283185307179586…

In [9]:
# t = 0
dt = 0.001
dq= np.zeros(robot.nq)

#q = (q_min+q_max)/2
q_c = q
robot.rebuildData()

tau_m_max = 6 # amps
tau_ratio = 1.0


t0= time.time()
n_loop  = 0
torque_limit_time = 1.75

torque_limits = [1.0, 0.75, 0.5, 0.3, 0.2, 0.1, 0]
torque_limit_index = 0

t = 0
q_c_list = []
t_list = []

P = 1.2
I = 0.8
D = 0.05
e_sum = 0
e = 0
e_last = 0
while t < 15:
    t = t+dt

    # position control strategies 
    # strategy no.1
    if n_loop and (n_loop % np.round(torque_limit_time/dt) == 0):
        tau_ratio = torque_limits[torque_limit_index]
        torque_limit_index = torque_limit_index+1
        torque_limit_index = np.clip(torque_limit_index, 0, len(torque_limits)-1)
        print(tau_ratio)

    # strategy no.2
    #tau_ratio = np.clip(0.5*np.max(dq)/dq_max, 0, 0.75)

    # pids
    e_last = e
    e = JW@(q-q_c)
    e_sum  = e_sum+e*dt
    de = (e - e_last)/dt
    tau_c = (P*e + I*e_sum + D*de)@W  # + compute_gravity(robot, q_c)
    tau_m = np.clip(tau_c@W_inv, -tau_ratio*tau_m_max,tau_ratio*tau_m_max)

    
    # torque control strategies 
    # strategy no.3
    #dq_m = JW_inv@dq
    #tau_m = np.clip(-350.0*dq_m, -tau_ratio*tau_m_max, tau_ratio*tau_m_max)

    
    # velocity control strategies 
    # strategy no.4
    tau_ratio = 1.0
    dq_m = JW@dq
    tau_m = np.clip(-0.2*dq_m, -tau_ratio*tau_m_max, tau_ratio*tau_m_max)
    
    tau_c = tau_m@W    # a bit of friction
    tau_c = tau_c - np.diag([1.5,1.5,1.0,1.0])@dq #more friction for the shoulder less for the elbow

    # carried object
    J,s = jacobian(robot, q_c, tip="r_arm_tip")
    tau_c = tau_c - J.T@np.array([0,0, 2.0*9.81]).T
    
    ddq = pin.aba(robot.model, robot.data, q_c, dq, tau_c)
    q_old = q_c
    q_c = np.clip(dq*dt + ddq*(dt**2)/2 + q_c, q_min,q_max)
    dq = (q_c - q_old)/dt#dq + ddq*dt
    if np.sum(np.isnan(q_c)):
        print("Nan")
        break
    n_loop = n_loop +1
    q_c_list.append(q_c)
    t_list.append(t)
    
print(t)
t_list = np.array(t_list)
# display
t0 = time.time()
while t_list[-1] > time.time()-t0:
    time.sleep(0.01)
    i = np.where(t_list > time.time()-t0)[0][0]
    viz.display(q_c_list[i])
print(time.time()-t0)


1.0
0.75
0.5
0.3
0.2
0.1
0
0
15.000999999997125


IndexError: index 0 is out of bounds for axis 0 with size 0

In [12]:
from python_client import PyPoulpeRemoteClient

id_shoulder = 1
id_elbow = 3

n_axis = 2

# Create an instance of the client
client_sh = PyPoulpeRemoteClient("http://127.0.0.1:50098", [id_shoulder], 0.001)
client_el = PyPoulpeRemoteClient("http://127.0.0.1:50098", [id_elbow], 0.001)

print(client_sh.get_connected_devices())

client_sh.set_torque_limit(id_shoulder,[1.0]*n_axis)
client_sh.set_velocity_limit(id_shoulder, [1.0]*n_axis)
client_el.set_torque_limit(id_elbow,[1.0]*n_axis)
client_el.set_velocity_limit(id_elbow, [1.0]*n_axis)

([6, 3, 4, 1, 0, 2, 5], ['LeftElbowOrbita2d', 'RightElbowOrbita2d', 'LeftShoulderOrbita2d', 'RightShoulderOrbita2d', 'NeckOrbita3d', 'RightWristOrbita3d', 'LeftWristOrbita3d'])


In [14]:
client_sh.set_torque_limit(id_shoulder,[1.0]*n_axis)
client_sh.set_velocity_limit(id_shoulder, [1.0]*n_axis)
client_el.set_torque_limit(id_elbow,[1.0]*n_axis)
client_el.set_velocity_limit(id_elbow, [1.0]*n_axis)

client_sh.turn_off(id_shoulder)
client_el.turn_off(id_elbow)

client_sh.set_mode_of_operation(id_shoulder,4)
client_el.set_mode_of_operation(id_elbow,4)

client_sh.get_mode_of_operation(id_shoulder), client_el.get_mode_of_operation(id_elbow)

(4, 4)

In [1399]:
#while True:
q = np.array([client_sh.get_position_actual_value(id_shoulder), client_el.get_position_actual_value(id_elbow)]).flatten()
viz.display(Jo_inv@(q)*q_dir -q_zeros)
time.sleep(0.01)

In [837]:
q = np.array([client_sh.get_position_actual_value(id_shoulder), client_el.get_position_actual_value(id_elbow)]).flatten()
q

array([ 3.94625401, -2.83545518,  2.05704308, -1.37100065])

In [19]:
axis_zeros = np.array([0, -np.pi/2,np.deg2rad(10),0])

In [20]:
q_dir = np.array([1,-1,1,1])

In [1014]:
Jo_inv@(q -q_zeros)

array([-1.22265517,  2.96677931,  0.27985296,  0.70292133])

In [1030]:
q

array([ 3.31383586, -3.94566822,  2.04180408, -0.80382991])

In [923]:
Jo_inv@(q) -q_zeros

array([-3.03857763,  0.21002658,  0.06131851,  0.8786544 ])

In [21]:
q_zeros = np.array([client_sh.get_axis_sensor_zeros(id_shoulder), client_el.get_axis_sensor_zeros(id_elbow)]).flatten()
q_zeros

array([5.53582764, 2.19050479, 5.39158297, 2.9338789 ])

In [23]:
q_a = np.array([client_sh.get_axis_sensors(id_shoulder), client_el.get_axis_sensors(id_elbow)]).flatten()
q = (q_a -q_zeros - axis_zeros)*q_dir
viz.display(q)

In [24]:
q_dir = np.array([1,-1,1,1])
t_dir = np.array([-1,1,-1,-1])

In [29]:
t0 = time.time()
n_loop = 0
while time.time() - t0 < 60:
    q_a = np.array([client_sh.get_axis_sensors(id_shoulder), client_el.get_axis_sensors(id_elbow)]).flatten()
    q = (q_a -q_zeros - axis_zeros)*q_dir
    viz.display(q)
    
    tau_ratio = 1.0
    tau_m_max = 6 # amps
    
    tau_axis = compute_gravity(robot, q)
    tau_m = np.clip(tau_axis*t_dir@W_inv, -tau_ratio*tau_m_max,tau_ratio*tau_m_max)
    client_sh.set_target_torque(id_shoulder, tau_m[:2]*1000)
    client_el.set_target_torque(id_elbow, tau_m[-2:]*1000)
    time.sleep(0.01)
    n_loop = n_loop+1
    if n_loop % 20 == 0:
        print(time.time()-t0)



print("done")
client_sh.set_target_torque(id_shoulder, [0.0,0.0])
client_el.set_target_torque(id_elbow, [0.0,0.0])

0.6736946105957031
1.3283987045288086
1.9659452438354492
2.5962822437286377
3.224740982055664
3.848586082458496
4.477980613708496
5.108226776123047
5.79599142074585
6.471286773681641
7.13000750541687
7.783382415771484
8.41500973701477
9.057157516479492
9.682555437088013
10.32374358177185
10.948745727539062
11.591250896453857
12.243139743804932
12.877249479293823
13.514568328857422
14.180142879486084
14.829013586044312
15.51229739189148
16.1724750995636
16.80583930015564
17.447632789611816
18.09385085105896
18.730907678604126
19.38313579559326
20.07638931274414
20.760868310928345
21.40785527229309
22.183581352233887
22.865943431854248
23.536691427230835
24.216312170028687
24.87681555747986
25.535370588302612
26.345669984817505
27.00716471672058
27.674405813217163
28.473681926727295
29.121670484542847
29.75924587249756
30.413925170898438
31.05926752090454
31.71383810043335
32.359734296798706
33.0057213306427
33.64107036590576
34.27371549606323
34.89593482017517
35.534289598464966
36.1755

In [1371]:

client_sh.set_target_torque(id_shoulder, [0.0,0.0])
client_el.set_target_torque(id_elbow, [500.0,500.0])

In [1404]:

client_sh.set_target_torque(id_shoulder, [0.0,0.0])
client_el.set_target_torque(id_elbow, [0.0,0.0])

In [25]:

client_sh.turn_on(id_shoulder)
client_el.turn_on(id_elbow)

In [1564]:
q_a = np.array([client_sh.get_axis_sensors(id_shoulder), client_el.get_axis_sensors(id_elbow)]).flatten()
q = (q_a -q_zeros - axis_zeros)*q_dir
viz.display(q)

tau_ratio = 1.0
tau_m_max = 4 # amps

tau_axis = compute_gravity(robot, q)
tau_m = np.clip(q_dir*tau_axis@W_inv, -tau_ratio*tau_m_max,tau_ratio*tau_m_max)
tau_m

array([ 0.00994224, -0.31766942, -0.23738661,  0.21651696])

In [1418]:
client_el.get_torque_state(id_elbow)

True

In [1419]:
client_el.turn_on(id_elbow)

In [1578]:
client_sh.set_torque_limit(id_shoulder,[1.0]*n_axis)
client_sh.set_velocity_limit(id_shoulder, [1.0]*n_axis)
client_el.set_torque_limit(id_elbow,[1.0]*n_axis)
client_el.set_velocity_limit(id_elbow, [1.0]*n_axis)

client_sh.turn_off(id_shoulder)
client_el.turn_off(id_elbow)

client_sh.set_mode_of_operation(id_shoulder,1)
client_el.set_mode_of_operation(id_elbow,1)

client_sh.get_mode_of_operation(id_shoulder), client_el.get_mode_of_operation(id_elbow)

(1, 1)

In [1582]:

client_sh.turn_on(id_shoulder)
client_el.turn_on(id_elbow)

In [1580]:

client_sh.turn_off(id_shoulder)
client_el.turn_off(id_elbow)